# 📊 Baseline Pipeline — Pandas Sequential
**Proyek:** SDG 10 Big Data Analytics — Medallion Architecture  
**Institut Teknologi Sumatera 2026**

Notebook ini mengimplementasikan **pipeline baseline berbasis Pandas** (sekuensial) untuk:
1. Ingesti & validasi data IPUMS Brazil-Mexico 2010
2. Pembersihan & transformasi data
3. Perhitungan metrik ketimpangan: Gini, Palma Ratio, Theil Index
4. Benchmark throughput & latensi (dibandingkan Spark di notebook 03)


## 0. Setup & Import Library

In [ ]:
import pandas as pd
import numpy as np
import time

DATA_PATH = '/home/jovyan/data/ipums_brazil_mexico_2010.csv'
COUNTRY_MAP = {76: 'Brazil', 484: 'Mexico'}
SENTINEL_INCTOT = [9999999, 99999999, 999999999]

# Step 1: Hitung total baris dengan cepat
print('🔢 Menghitung total baris...')
t0 = time.time()

with open(DATA_PATH, 'r') as f:
    total_rows = sum(1 for _ in f) - 1  # minus header

print(f'   Total baris : {total_rows:,} ({time.time()-t0:.1f} detik)')

n_sample = int(total_rows * 0.1)
print(f'   10% sampel  : {n_sample:,} baris')
print(f'   Ambil dari  : baris 1–{n_sample:,} (awal) + {total_rows-n_sample:,}–{total_rows:,} (akhir)')

In [ ]:
# Bronze Layer : Baca 10% Brazil + 10% Mexico 
t0 = time.time()

brazil_chunks = []
mexico_chunks = []

for chunk in pd.read_csv(DATA_PATH):

    # Filter Brazil
    br = chunk[chunk['COUNTRY'] == 76].copy()
    if len(br) > 0:
        br = br[~br['INCTOT'].isin(SENTINEL) & (br['INCTOT'] >= 0)]
        if len(br) > 0:
            brazil_chunks.append(br.sample(frac=SAMPLE_FRAC, random_state=42))

    # Filter Mexico
    mx = chunk[chunk['COUNTRY'] == 484].copy()
    if len(mx) > 0:
        mx = mx[~mx['INCTOT'].isin(SENTINEL) & (mx['INCTOT'] >= 0)]
        if len(mx) > 0:
            mexico_chunks.append(mx.sample(frac=SAMPLE_FRAC, random_state=42))

# Gabungkan
parts = []
if brazil_chunks:
    df_brazil = pd.concat(brazil_chunks, ignore_index=True)
    parts.append(df_brazil)
    print(f'🇧🇷 Brazil  : {len(df_brazil):,} baris (10% sampel)')
else:
    print('🇧🇷 Brazil  : ❌ tidak ditemukan')

if mexico_chunks:
    df_mexico = pd.concat(mexico_chunks, ignore_index=True)
    parts.append(df_mexico)
    print(f'🇲🇽 Mexico  : {len(df_mexico):,} baris (10% sampel)')
else:
    print('🇲🇽 Mexico  : ❌ tidak ditemukan dalam file')

df_bronze = pd.concat(parts, ignore_index=True)
df_bronze['country_name'] = df_bronze['COUNTRY'].map(COUNTRY_MAP)

t_bronze = time.time() - t0

print(f'\n📦 Total gabungan : {len(df_bronze):,} baris')
print(f'⏱  Waktu          : {t_bronze:.2f} detik')
print(f'⚡ Throughput      : {len(df_bronze)/t_bronze:,.0f} baris/detik')
print(f'💾 Memory          : {df_bronze.memory_usage(deep=True).sum()/1e6:.1f} MB')

## 1. Bronze Layer — Ingesti & Validasi Skema

In [ ]:
# ── Timer ingesti ─────────────────────────────────────────────────────────────
t0 = time.time()

df_bronze = pd.read_csv(
    DATA_PATH,
    dtype={
        'COUNTRY'  : 'int32',
        'YEAR'     : 'int16',
        'SAMPLE'   : 'int64',
        'SERIAL'   : 'int64',
        'HHWT'     : 'float32',
        'PERNUM'   : 'int16',
        'PERWT'    : 'float32',
        'AGE'      : 'int16',
        'SEX'      : 'int8',
        'EDATTAIN' : 'int8',
        'EDATTAIND': 'int16',
        'EMPSTAT'  : 'int8',
        'EMPSTATD' : 'int16',
        'OCCISCO'  : 'int16',
        'INDGEN'   : 'int16',
        'INCTOT'   : 'int64',
        'INCEARN'  : 'int64',
    }
)

t_ingest = time.time() - t0

print(f'⏱  Waktu ingesti     : {t_ingest:.2f} detik')
print(f'📦 Total baris       : {len(df_bronze):,}')
print(f'📋 Total kolom       : {len(df_bronze.columns)}')
print(f'💾 Memory usage      : {df_bronze.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'⚡ Throughput        : {len(df_bronze)/t_ingest:,.0f} baris/detik')

In [ ]:
# ── Tampilkan schema & sampel ─────────────────────────────────────────────────
print('=== SCHEMA ===')
print(df_bronze.dtypes)
print('\n=== SAMPEL 5 BARIS ===')
df_bronze.head()

In [ ]:
# ── Distribusi per negara ──────────────────────────────────────────────────────
country_counts = df_bronze['COUNTRY'].map(COUNTRY_MAP).value_counts()
print('=== DISTRIBUSI NEGARA ===')
print(country_counts.to_string())

# Statistik dasar
print('\n=== STATISTIK DESKRIPTIF (INCTOT, INCEARN, AGE, PERWT) ===')
df_bronze[['INCTOT','INCEARN','AGE','PERWT']].describe()

## 2. Silver Layer — Cleaning & Transformasi

In [ ]:
t0 = time.time()

df_silver = df_bronze.copy()

# ── 2a. Hapus nilai sentinel / missing ────────────────────────────────────────
before = len(df_silver)
df_silver = df_silver[
    (df_silver['INCTOT']  != MISSING_INCTOT) &
    (df_silver['INCEARN'] != MISSING_INCEARN) &
    (df_silver['INCTOT']  >= 0) &
    (df_silver['AGE']     >= 15) &   # usia kerja
    (df_silver['AGE']     <= 65)
]
after_filter = len(df_silver)

# ── 2b. Drop duplikat ─────────────────────────────────────────────────────────
df_silver = df_silver.drop_duplicates(subset=['SAMPLE','SERIAL','PERNUM'])
after_dedup = len(df_silver)

# ── 2c. Normalisasi PPP (faktor konversi 2010) ────────────────────────────────
# Brazil: 1 BRL = ~0.5664 USD PPP | Mexico: 1 MXN = ~0.6838 USD PPP
PPP_FACTOR = {76: 0.5664, 484: 0.6838}
df_silver['income_ppp'] = df_silver.apply(
    lambda r: r['INCTOT'] * PPP_FACTOR.get(r['COUNTRY'], 1.0), axis=1
)

# ── 2d. Label kategorikal ─────────────────────────────────────────────────────
df_silver['country_name'] = df_silver['COUNTRY'].map(COUNTRY_MAP)
df_silver['sex_label']    = df_silver['SEX'].map({1: 'Laki-laki', 2: 'Perempuan'})
df_silver['empstat_label']= df_silver['EMPSTAT'].map({
    1: 'Employed', 2: 'Unemployed', 3: 'Not in labor force'
})

# ── 2e. Bin usia ──────────────────────────────────────────────────────────────
df_silver['age_group'] = pd.cut(
    df_silver['AGE'],
    bins=[14, 24, 34, 44, 54, 65],
    labels=['15-24','25-34','35-44','45-54','55-65']
)

t_silver = time.time() - t0

print(f'⏱  Waktu Silver layer : {t_silver:.2f} detik')
print(f'📦 Baris sebelum filter   : {before:,}')
print(f'📦 Setelah filter sentinel: {after_filter:,} (−{before-after_filter:,})')
print(f'📦 Setelah dedup          : {after_dedup:,} (−{after_filter-after_dedup:,})')
print(f'📊 Kolom baru             : income_ppp, country_name, sex_label, empstat_label, age_group')

In [ ]:
# Cek hasil Silver layer
df_silver[['country_name','AGE','age_group','sex_label','empstat_label','INCTOT','income_ppp','PERWT']].head(10)

## 3. Gold Layer — Metrik Ketimpangan

In [ ]:
# ── Fungsi metrik ketimpangan ─────────────────────────────────────────────────

def gini_coefficient(income: np.ndarray, weights: np.ndarray = None) -> float:
    """Hitung Gini coefficient dengan pembobotan (PERWT)."""
    if weights is None:
        weights = np.ones(len(income))
    # Urutkan berdasarkan income
    order = np.argsort(income)
    income  = income[order]
    weights = weights[order]
    # Cumulative weighted income share
    cum_w = np.cumsum(weights)
    cum_y = np.cumsum(income * weights)
    total_w = cum_w[-1]
    total_y = cum_y[-1]
    if total_y == 0:
        return 0.0
    # Formula Gini via area under Lorenz curve
    B = np.sum((cum_y[:-1] / total_y) * (weights[1:] / total_w))
    return 1 - 2 * B


def palma_ratio(income: np.ndarray, weights: np.ndarray = None) -> float:
    """Palma ratio = share top 10% / share bottom 40%."""
    if weights is None:
        weights = np.ones(len(income))
    order = np.argsort(income)
    income  = income[order]
    weights = weights[order]
    cum_w = np.cumsum(weights)
    total_w = cum_w[-1]
    p40_mask  = cum_w <= 0.40 * total_w
    p90_mask  = cum_w >= 0.90 * total_w
    total_y   = np.sum(income * weights)
    share_b40 = np.sum(income[p40_mask] * weights[p40_mask]) / total_y
    share_t10 = np.sum(income[p90_mask] * weights[p90_mask]) / total_y
    return share_t10 / share_b40 if share_b40 > 0 else np.nan


def theil_index(income: np.ndarray, weights: np.ndarray = None) -> float:
    """Theil T index (weighted)."""
    if weights is None:
        weights = np.ones(len(income))
    mask = income > 0
    income  = income[mask]
    weights = weights[mask]
    mu = np.average(income, weights=weights)
    ratio = income / mu
    return float(np.average(ratio * np.log(ratio), weights=weights))


print('✅ Fungsi metrik ketimpangan siap!')

In [ ]:
t0 = time.time()

# ── Hitung metrik per negara ──────────────────────────────────────────────────
results = []
for country, grp in df_silver.groupby('country_name'):
    inc = grp['income_ppp'].values.astype(float)
    wt  = grp['PERWT'].values.astype(float)
    
    # Kuintil (Q1-Q5)
    quantiles = np.quantile(np.repeat(inc, wt.astype(int).clip(1)), [0.2,0.4,0.6,0.8,1.0])
    
    results.append({
        'Negara'         : country,
        'N_individu'     : len(grp),
        'N_weighted'     : int(wt.sum()),
        'Median_Inc_PPP' : round(np.median(inc), 2),
        'Mean_Inc_PPP'   : round(np.average(inc, weights=wt), 2),
        'Gini'           : round(gini_coefficient(inc, wt), 4),
        'Palma_Ratio'    : round(palma_ratio(inc, wt), 4),
        'Theil_T'        : round(theil_index(inc, wt), 4),
        'Q1_batas'       : round(quantiles[0], 2),
        'Q3_batas'       : round(quantiles[2], 2),
        'Q5_batas'       : round(quantiles[4], 2),
    })

df_gold = pd.DataFrame(results)
t_gold = time.time() - t0

print(f'⏱  Waktu Gold layer : {t_gold:.2f} detik')
print('\n=== HASIL METRIK KETIMPANGAN ===')
df_gold

In [ ]:
# ── Shared Prosperity Premium (bottom 40% vs rata-rata) ───────────────────────
print('=== SHARED PROSPERITY PREMIUM ===')
for country, grp in df_silver.groupby('country_name'):
    inc = grp['income_ppp'].values.astype(float)
    wt  = grp['PERWT'].values.astype(float)
    mean_all = np.average(inc, weights=wt)
    # Bottom 40%
    order = np.argsort(inc)
    cum_w = np.cumsum(wt[order])
    b40_mask = cum_w <= 0.40 * cum_w[-1]
    mean_b40 = np.average(inc[order][b40_mask], weights=wt[order][b40_mask])
    premium  = ((mean_b40 / mean_all) - 1) * 100
    flag     = '🔴 Anomali' if mean_b40 < 0.5 * mean_all else '🟢 Normal'
    print(f'  {country:8s} | Mean all: {mean_all:8.2f} | Mean B40: {mean_b40:8.2f} | Premium: {premium:+.1f}% {flag}')

## 4. Visualisasi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Metrik Ketimpangan Pendapatan — Brazil vs Mexico 2010', fontsize=14, fontweight='bold')

colors = ['#2196F3', '#FF5722']
countries = df_gold['Negara'].tolist()

# Gini
axes[0].bar(countries, df_gold['Gini'], color=colors)
axes[0].set_title('Gini Coefficient')
axes[0].set_ylim(0, 1)
axes[0].axhline(0.4, color='red', linestyle='--', alpha=0.5, label='Threshold tinggi (0.4)')
axes[0].legend(fontsize=8)
for i, v in enumerate(df_gold['Gini']):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# Palma Ratio
axes[1].bar(countries, df_gold['Palma_Ratio'], color=colors)
axes[1].set_title('Palma Ratio (Top10% / Bottom40%)')
for i, v in enumerate(df_gold['Palma_Ratio']):
    axes[1].text(i, v + 0.05, f'{v:.4f}', ha='center', fontweight='bold')

# Theil T
axes[2].bar(countries, df_gold['Theil_T'], color=colors)
axes[2].set_title('Theil Index (T)')
for i, v in enumerate(df_gold['Theil_T']):
    axes[2].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/home/jovyan/data/gold_metrics_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot tersimpan!')

In [ ]:
# ── Lorenz Curve ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='Perfect equality')

for (country, grp), color in zip(df_silver.groupby('country_name'), colors):
    inc = grp['income_ppp'].values.astype(float)
    wt  = grp['PERWT'].values.astype(float)
    order = np.argsort(inc)
    inc = inc[order]; wt = wt[order]
    cum_w = np.cumsum(wt) / wt.sum()
    cum_y = np.cumsum(inc * wt) / (inc * wt).sum()
    ax.plot(cum_w, cum_y, color=color, label=country, linewidth=2)

ax.set_xlabel('Kumulatif populasi')
ax.set_ylabel('Kumulatif pendapatan')
ax.set_title('Lorenz Curve — Brazil vs Mexico 2010')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/home/jovyan/data/lorenz_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Benchmark Summary

In [ ]:
total_rows = len(df_bronze)
total_time = t_ingest + t_silver + t_gold

print('=' * 55)
print('        BENCHMARK SUMMARY — BASELINE PANDAS')
print('=' * 55)
print(f'  Total baris data       : {total_rows:>12,}')
print(f'  Waktu ingesti (Bronze) : {t_ingest:>10.2f} detik')
print(f'  Waktu cleaning (Silver): {t_silver:>10.2f} detik')
print(f'  Waktu agregasi (Gold)  : {t_gold:>10.2f} detik')
print(f'  ─────────────────────────────────────────')
print(f'  Total waktu end-to-end : {total_time:>10.2f} detik')
print(f'  Throughput ingesti     : {total_rows/t_ingest:>10,.0f} baris/dtk')
print(f'  Throughput total       : {total_rows/total_time:>10,.0f} baris/dtk')
print('=' * 55)
print('  ✅ Simpan angka ini untuk dibandingkan di notebook 03!')

In [ ]:
# ── Simpan hasil Gold ke CSV ──────────────────────────────────────────────────
df_gold.to_csv('/home/jovyan/data/gold_metrics_baseline.csv', index=False)
df_silver.to_parquet('/home/jovyan/data/silver_baseline.parquet', index=False)
print('✅ Gold metrics  → /home/jovyan/data/gold_metrics_baseline.csv')
print('✅ Silver data   → /home/jovyan/data/silver_baseline.parquet')